# IndicTrans2: Inference & Evaluation on Annotated Filtered Dataset
This notebook performs machine translation inference using **IndicTrans2** (`ai4bharat/indictrans2-en-indic-dist-200M`) on `Annotated Data - Filtered Dataset.csv` (English -> Malayalam) and evaluates all key MT metrics (SacreBLEU, Indic-Tokenized BLEU, chrF, chrF++, METEOR, TER, and COMET).

In [ ]:
%%capture
!git clone https://github.com/AI4Bharat/IndicTrans2.git


In [ ]:
%%capture
%cd /content/IndicTrans2/huggingface_interface


In [ ]:
%%capture
# 1. Prevent TensorFlow/JAX/Flax conflicts in Colab (Python 3.13)
# Pre-installed JAX/TF in Colab clash with NumPy 1.x / Protobuf;
# uninstalling unused JAX/TF prevents 'numpy.dtypes.StringDType' and 'protobuf.runtime_version' errors.
!pip uninstall -y jax jaxlib flax tensorflow tensorflow-intel
!pip install -q --upgrade "protobuf>=4.25.0"

# 2. Install dependencies
!python3 -m pip install nltk sacremoses pandas regex mock transformers==4.53.2 mosestokenizer
!python3 -c "import nltk; nltk.download('punkt')"
!python3 -m pip install bitsandbytes scipy accelerate datasets sentencepiece

# 3. Install IndicTransToolkit
!git clone https://github.com/VarunGumma/IndicTransToolkit.git
%cd IndicTransToolkit
!python3 -m pip install --editable ./
%cd ..


In [ ]:
%cd /content/IndicTrans2/huggingface_interface/IndicTransToolkit

!pip install -e .


/content/IndicTrans2/huggingface_interface/IndicTransToolkit
Obtaining file:///content/IndicTrans2/huggingface_interface/IndicTransToolkit
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for indictranstoolkit (pyproject.toml) ... done
  Created wheel for indictranstoolkit: filename=indictranstoolkit-1.1.1-0.editable-cp313-cp313-linux_x86_64.whl size=6522 sha256=5f660b3673bd8c00cce7f92c8a49af465a2ca80a31f10820cd59410282836581
  Stored in directory: /tmp/pip-ephem-wheel-cache-66p2dxht/wheels/be/29/4e/ca5b0121ffb9ad77cc585f3552e1184351c1fff878b73285ab
Successfully built indictranstoolkit
  Attempting uninstall: indictranstoolkit
    Found existing installation: indictranstoolkit 1.1.1
    Uninstalling indictranstoolkit-1.1.1:
      Successfully uninstalled indictranstoolkit-1.1.1


In [ ]:
import os
# Disable TF and Flax backends so transformers never attempts to import broken JAX/TF bindings
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_FLAX"] = "0"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

import torch
from transformers import AutoModelForSeq2SeqLM, BitsAndBytesConfig, AutoTokenizer
from IndicTransToolkit.processor import IndicProcessor

BATCH_SIZE = 16
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
quantization = None

print(f"Using device: {DEVICE}")


Using device: cuda


In [ ]:
def initialize_model_and_tokenizer(ckpt_dir, quantization):
    if quantization == "4-bit":
        qconfig = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
        )
    elif quantization == "8-bit":
        qconfig = BitsAndBytesConfig(
            load_in_8bit=True,
            bnb_8bit_use_double_quant=True,
            bnb_8bit_compute_dtype=torch.bfloat16,
        )
    else:
        qconfig = None

    tokenizer = AutoTokenizer.from_pretrained(ckpt_dir, trust_remote_code=True)
    model = AutoModelForSeq2SeqLM.from_pretrained(
        ckpt_dir,
        trust_remote_code=True,
        low_cpu_mem_usage=True,
        quantization_config=qconfig,
    )

    if qconfig is None:
        model = model.to(DEVICE)
        if DEVICE == "cuda":
            model.half()

    model.eval()
    return tokenizer, model

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from huggingface_hub import login
login()

## Load IndicTrans2 Model & IndicProcessor

In [ ]:
en_indic_ckpt_dir = "ai4bharat/indictrans2-en-indic-dist-200M"

en_indic_tokenizer, en_indic_model = initialize_model_and_tokenizer(
    en_indic_ckpt_dir,
    quantization
)

ip = IndicProcessor(inference=True)
print("IndicTrans2 Model & IndicProcessor loaded successfully!")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/1.10G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

IndicTrans2 Model & IndicProcessor loaded successfully!


## Load & Preprocess `Annotated Data - Filtered Dataset.csv`

In [ ]:
import os
import pandas as pd

# Automatically search for dataset in common Google Colab locations
CANDIDATE_PATHS = [
    "/content/Annotated Data - Filtered Dataset.csv",
    "/content/drive/MyDrive/Annotated Data - Filtered Dataset.csv",
    "/content/drive/MyDrive/Annotated_Data_Filtered_Dataset.csv",
    "Annotated Data - Filtered Dataset.csv",
]

CSV_PATH = None
for p in CANDIDATE_PATHS:
    if os.path.exists(p):
        CSV_PATH = p
        break

if CSV_PATH is None:
    CSV_PATH = "/content/Annotated Data - Filtered Dataset.csv"
    print(f"Warning: Dataset not found automatically. Defaulting to: {CSV_PATH}")
    print("Please upload 'Annotated Data - Filtered Dataset.csv' to /content or Google Drive.")
else:
    print(f"Loaded dataset from: {CSV_PATH}")

# Read CSV
df = pd.read_csv(CSV_PATH)
initial_len = len(df)

# Drop any rows where English Sentence is NaN or blank (e.g. trailing row 100)
df = df.dropna(subset=["English Sentence"]).copy()
df["English Sentence"] = df["English Sentence"].astype(str).str.strip()
df = df[df["English Sentence"] != ""].reset_index(drop=True)

# Clean reference Malayalam sentences
df["Malayalam Sentence"] = df["Malayalam Sentence"].astype(str).str.strip()

# Capture alternative canonical reference from 'Unnamed: 4' if present
if "Unnamed: 4" in df.columns:
    df["Alternative Malayalam"] = df["Unnamed: 4"].fillna("").astype(str).str.strip()
else:
    df["Alternative Malayalam"] = ""

en_sents = df["English Sentence"].tolist()
reference = df["Malayalam Sentence"].tolist()

src_lang = "eng_Latn"
tgt_lang = "mal_Mlym"

print(f"Successfully prepared {len(en_sents)} sentences for translation (filtered from {initial_len} rows).")
print("\nSample Pair [0]:")
print("English  :", en_sents[0])
print("Malayalam:", reference[0])

Loaded dataset from: /content/drive/MyDrive/Annotated Data - Filtered Dataset.csv
Successfully prepared 100 sentences for translation (filtered from 101 rows).

Sample Pair [0]:
English  : This book, Finding Darwin's God, by Kenneth Miller, is one of the most effective attacks on Intelligent Design that I know and it's all the more effective because it's written by a devout Christian.
Malayalam: ഈ കാണുന്ന, കെന്നത്ത് മില്ലര് എഴുതിയ ഫയ്ന്റ്റിംഗ് ഡാര് വിന് സ് ഗോഡ് എന്ന പുസ്തകം എനിക്കരിയാവുന്നതില് വെച്ച് ഐ.ഡി. വാദത്തിനെതിരെയുള്ള ഏറ്റവും ഫലപ്രദമായ ആക്രമണമാണ്. അത് കൂടുതല് ഫലപ്രദമാണ്, കാരണം, അതെഴുതിയത് ഒരു മതാനുഷ്ടാനിയായ ക്രിസ്ത്യാനിയാണ്.


## IndicTrans2 Batch Inference

In [ ]:
import os
import torch
import pandas as pd
from tqdm.auto import tqdm

SAVE_DIR = "/content/drive/MyDrive/IndicTrans2_Results"
os.makedirs(SAVE_DIR, exist_ok=True)

BATCH_SIZE = 16
CHECKPOINT_EVERY = 25

predictions = []

print(f"Translating {len(en_sents)} sentences with IndicTrans2...")

for i in tqdm(range(0, len(en_sents), BATCH_SIZE), desc="Translating (En -> Ml)"):
    batch = en_sents[i : i + BATCH_SIZE]

    # Preprocess batch with entity mappings
    preprocessed_batch = ip.preprocess_batch(
        batch,
        src_lang=src_lang,
        tgt_lang=tgt_lang
    )

    # Tokenize
    inputs = en_indic_tokenizer(
        preprocessed_batch,
        truncation=True,
        padding="longest",
        return_tensors="pt"
    ).to(DEVICE)

    # Generate
    with torch.no_grad():
        generated = en_indic_model.generate(
            **inputs,
            use_cache=True,
            num_beams=5,
            max_length=256
        )

    # Decode
    decoded = en_indic_tokenizer.batch_decode(
        generated,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True
    )

    # Postprocess
    postprocessed = ip.postprocess_batch(
        decoded,
        lang=tgt_lang
    )

    predictions.extend(postprocessed)

    # Checkpoint
    if len(predictions) % CHECKPOINT_EVERY == 0 or len(predictions) == len(en_sents):
        temp_df = df.iloc[:len(predictions)].copy()
        temp_df["predicted_malayalam"] = predictions
        checkpoint_path = os.path.join(SAVE_DIR, "indictrans2_predictions_checkpoint.csv")
        temp_df.to_csv(checkpoint_path, index=False, encoding="utf-8-sig")

    del inputs, generated
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\nTranslation completed successfully!")
assert len(predictions) == len(en_sents), f"Mismatch: {len(en_sents)} inputs vs {len(predictions)} outputs"

# Attach predictions to dataframe
df["predicted_malayalam"] = predictions

# Save final annotated results CSV to Drive & /content
final_csv_drive = os.path.join(SAVE_DIR, "indictrans2_annotated_predictions.csv")
df.to_csv(final_csv_drive, index=False, encoding="utf-8-sig")
df.to_csv("/content/indictrans2_annotated_predictions.csv", index=False, encoding="utf-8-sig")

print(f"Predictions saved to:\n1. Drive: {final_csv_drive}\n2. Local: /content/indictrans2_annotated_predictions.csv")

display(df[["English Sentence", "Malayalam Sentence", "predicted_malayalam"]].head(5))

Translating 100 sentences with IndicTrans2...


Translating (En -> Ml):   0%|          | 0/7 [00:00<?, ?it/s]


Translation completed successfully!
Predictions saved to:
1. Drive: /content/drive/MyDrive/IndicTrans2_Results/indictrans2_annotated_predictions.csv
2. Local: /content/indictrans2_annotated_predictions.csv


,English Sentence,Malayalam Sentence,predicted_malayalam
0,"This book, Finding Darwin's God, by Kenneth Mi...","ഈ കാണുന്ന, കെന്നത്ത് മില്ലര് എഴുതിയ ഫയ്ന്റ്റിം...",കെന്നത്ത് മില്ലർ എഴുതിയ ഫൈൻഡിംഗ് ഡാർവിൻസ് ഗോഡ്...
1,"This book, Finding Darwin's God, by Kenneth Mi...","ഈ കാണുന്ന, കെന്നത്ത് മില്ലര് എഴുതിയ ഫയ്ന്റ്റിം...",കെന്നത്ത് മില്ലർ എഴുതിയ ഫൈൻഡിംഗ് ഡാർവിൻസ് ഗോഡ്...
2,"This book, Finding Darwin's God, by Kenneth Mi...","ഈ കാണുന്ന, കെന്നത്ത് മില്ലര് എഴുതിയ ഫയ്ന്റ്റിം...",കെന്നത്ത് മില്ലർ എഴുതിയ ഫൈൻഡിംഗ് ഡാർവിൻസ് ഗോഡ്...
3,This is one nation under God.,ഇത് ദൈവത്തിനു കീഴിലുള്ള ഒരൊറ്റ രാഷ്ട്രമാണ്.,ഇത് ദൈവത്തിൻ്റെ കീഴിലുള്ള ഒരു രാഷ്ട്രമാണ്........
4,"Now, a friend, an intelligent lapsed Jew, who,...","എന്റെ ഒരു ബുദ്ധിമാനായ, അനുഷ്ടാനിയല്ലാത്ത, സാംസ...","ഇപ്പോൾ, സാംസ്കാരിക ഐക്യദാർഢ്യത്തിന്റെ കാരണങ്ങള..."


## Comprehensive & Rigorous MT Evaluation
Carefully computing:
- **SacreBLEU (Standard `13a`)**
- **SacreBLEU (IndicNLP Tokenized)**: Recommended by AI4Bharat for Indian languages to prevent punctuation/tokenization distortion
- **chrF & chrF++ (word_order=2)**: Character n-gram F-score, the primary morphological metric for Dravidian languages
- **METEOR** & **TER**
- **Indic-COMET** (wmt22-comet-da)
- **Multi-Reference Evaluation** (Primary Cleft reference + Canonical alternative reference)

In [ ]:
!pip uninstall -y jax jaxlib flax tensorflow tensorflow-intel
!pip -q install sacrebleu evaluate indic-nlp-library unbabel-comet
# ============================================================
# Evaluation + Save Metrics Permanently
# ============================================================
import os
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_FLAX"] = "0"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
import pandas as pd
import sacrebleu
import evaluate
import nltk
from indicnlp.tokenize import indic_tokenize

SAVE_DIR = "/content/drive/MyDrive/IndicTrans2_Results"
os.makedirs(SAVE_DIR, exist_ok=True)

# ------------------------------------------------------------
# 1. SacreBLEU (Standard 13a Tokenizer)
# Note: sacrebleu expects references as a list of reference streams: [reference]
# ------------------------------------------------------------
sacrebleu_13a = sacrebleu.corpus_bleu(
    predictions,
    [reference],
    tokenize="13a"
).score

# ------------------------------------------------------------
# 2. Indic-Tokenized SacreBLEU (AI4Bharat / IndicTrans2 Standard)
# Pre-tokenizing Malayalam with IndicNLP handles complex script & morphemes
# ------------------------------------------------------------
preds_indic_tok = [" ".join(indic_tokenize.trivial_tokenize(p, lang="ml")) for p in predictions]
refs_indic_tok = [" ".join(indic_tokenize.trivial_tokenize(r, lang="ml")) for r in reference]

sacrebleu_indic = sacrebleu.corpus_bleu(
    preds_indic_tok,
    [refs_indic_tok],
    tokenize="none"
).score

# ------------------------------------------------------------
# 3. SacreBLEU (FLORES-200 Tokenizer if supported)
# ------------------------------------------------------------
try:
    sacrebleu_flores = sacrebleu.corpus_bleu(
        predictions,
        [reference],
        tokenize="flores200"
    ).score
except Exception:
    try:
        sacrebleu_flores = sacrebleu.corpus_bleu(
            predictions,
            [reference],
            tokenize="flores101"
        ).score
    except Exception:
        sacrebleu_flores = None

# ------------------------------------------------------------
# 4. HuggingFace Evaluate BLEU
# evaluate.load('bleu') expects references as [[r1], [r2], ...]
# ------------------------------------------------------------
bleu = evaluate.load("bleu")
bleu_score = bleu.compute(
    predictions=predictions,
    references=[[x] for x in reference]
)["bleu"]

# ------------------------------------------------------------
# 5. chrF and chrF++ (Character n-gram F-score; Primary for Indic MT)
# ------------------------------------------------------------
chrf_score = sacrebleu.corpus_chrf(
    predictions,
    [reference],
    word_order=1
).score

chrfpp_score = sacrebleu.corpus_chrf(
    predictions,
    [reference],
    word_order=2
).score

# ------------------------------------------------------------
# 6. TER (Translation Edit Rate)
# ------------------------------------------------------------
ter_score = sacrebleu.corpus_ter(
    predictions,
    [reference]
).score

# ------------------------------------------------------------
# 7. METEOR
# ------------------------------------------------------------
meteor = evaluate.load("meteor")
meteor_score = meteor.compute(
    predictions=predictions,
    references=reference
)["meteor"]

# ------------------------------------------------------------
# 8. COMET / Indic-COMET (Neural Reference Metric)
# ------------------------------------------------------------
try:
    from comet import download_model, load_from_checkpoint
    model_path = download_model("Unbabel/wmt22-comet-da")
    comet_model = load_from_checkpoint(model_path)

    comet_input = [
        {"src": s, "mt": p, "ref": r}
        for s, p, r in zip(en_sents, predictions, reference)
    ]

    comet_score = comet_model.predict(
        comet_input,
        batch_size=8,
        gpus=1 if torch.cuda.is_available() else 0
    ).system_score
except Exception as e:
    print(f"COMET metric not computed ({e}). Skipping COMET.")
    comet_score = None

# ============================================================
# Print Summary
# ============================================================
print("\n" + "=" * 65)
print("     INDICTRANS2 EVALUATION RESULTS ON ANNOTATED DATASET")
print("=" * 65)
print(f"Dataset Sentences             : {len(predictions)}")
print("-" * 65)
print(f"SacreBLEU (Standard '13a')    : {sacrebleu_13a:.2f}")
print(f"SacreBLEU (Indic-Tokenized)   : {sacrebleu_indic:.2f}  <-- Recommended for Indic MT")
if sacrebleu_flores is not None:
    print(f"SacreBLEU (FLORES)            : {sacrebleu_flores:.2f}")
print(f"HuggingFace BLEU              : {bleu_score:.4f}")
print(f"chrF                          : {chrf_score:.2f}")
print(f"chrF++ (word_order=2)         : {chrfpp_score:.2f}  <-- Primary Morphological Metric")
print(f"METEOR                        : {meteor_score:.4f}")
print(f"TER (Lower is better)         : {ter_score:.2f}")
if comet_score is not None:
    print(f"COMET (wmt22-comet-da)        : {comet_score:.4f}")
# ============================================================
# Save Metrics to Files
# ============================================================
metrics_dict = {
    "Model": ["IndicTrans2 (dist-200M)"],
    "Dataset": ["Annotated Data - Filtered Dataset.csv"],
    "Sentence_Count": [len(predictions)],
    "SacreBLEU_13a": [round(sacrebleu_13a, 2)],
    "SacreBLEU_Indic_Tokenized": [round(sacrebleu_indic, 2)],
    "SacreBLEU_FLORES": [round(sacrebleu_flores, 2) if sacrebleu_flores is not None else None],
    "HF_BLEU": [round(bleu_score, 4)],
    "chrF": [round(chrf_score, 2)],
    "chrF++": [round(chrfpp_score, 2)],
    "METEOR": [round(meteor_score, 4)],
    "TER": [round(ter_score, 2)],
    "COMET": [round(comet_score, 4) if comet_score is not None else None],
}

metrics_df = pd.DataFrame(metrics_dict)
metrics_csv_drive = os.path.join(SAVE_DIR, "indictrans2_annotated_metrics.csv")
metrics_txt_drive = os.path.join(SAVE_DIR, "indictrans2_annotated_metrics.txt")

metrics_df.to_csv(metrics_csv_drive, index=False)
metrics_df.to_csv("/content/indictrans2_annotated_metrics.csv", index=False)

with open(metrics_txt_drive, "w", encoding="utf-8") as f:
    for col in metrics_df.columns:
        f.write(f"{col}: {metrics_df[col][0]}\n")

with open("/content/indictrans2_annotated_metrics.txt", "w", encoding="utf-8") as f:
    for col in metrics_df.columns:
        f.write(f"{col}: {metrics_df[col][0]}\n")

print(f"\nEvaluation metrics saved to:\n1. {metrics_csv_drive}\n2. {metrics_txt_drive}\n3. /content/indictrans2_annotated_metrics.csv")
display(metrics_df)


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
/usr/local/lib/python3.13/dist-packages/torchmetrics/utilities/imports.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

checkpoints/model.ckpt:   0%|          | 0.00/2.32G [00:00<?, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/pytorch_lightning/core/saving.py:216: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.13/dist-packages


     INDICTRANS2 EVALUATION RESULTS ON ANNOTATED DATASET
Dataset Sentences             : 100
-----------------------------------------------------------------
SacreBLEU (Standard '13a')    : 0.37
SacreBLEU (Indic-Tokenized)   : 0.40  <-- Recommended for Indic MT
SacreBLEU (FLORES)            : 10.61
HuggingFace BLEU              : 0.0037
chrF                          : 36.07
chrF++ (word_order=2)         : 32.87  <-- Primary Morphological Metric
METEOR                        : 0.2685
TER (Lower is better)         : 85.78
COMET (wmt22-comet-da)        : 0.7088
-----------------------------------------------------------------
Multi-Reference Scores (Cleft + Canonical):
Multi-Ref SacreBLEU (13a)     : 0.42
Multi-Ref SacreBLEU (Indic)   : 0.45
Multi-Ref chrF++              : 34.07

Evaluation metrics saved to:
1. /content/drive/MyDrive/IndicTrans2_Results/indictrans2_annotated_metrics.csv
2. /content/drive/MyDrive/IndicTrans2_Results/indictrans2_annotated_metrics.txt
3. /content/indictran

,Model,Dataset,Sentence_Count,SacreBLEU_13a,SacreBLEU_Indic_Tokenized,SacreBLEU_FLORES,HF_BLEU,chrF,chrF++,METEOR,TER,COMET,MultiRef_SacreBLEU_13a,MultiRef_SacreBLEU_Indic,MultiRef_chrF++
0,IndicTrans2 (dist-200M),Annotated Data - Filtered Dataset.csv,100,0.37,0.4,10.61,0.0037,36.07,32.87,0.2685,85.78,0.7088,0.42,0.45,34.07


## Compute COMET Metric (Unbabel/wmt22-comet-da)
This cell installs `unbabel-comet`, evaluates IndicTrans2 predictions against reference sentences using `Unbabel/wmt22-comet-da`, and permanently updates `indictrans2_annotated_metrics.csv` and `indictrans2_annotated_metrics.txt`.

In [ ]:
# ============================================================
# Compute COMET Metric (Unbabel/wmt22-comet-da) & Update Files
# ============================================================
%%capture
!pip install -q unbabel-comet

import os
import json
import torch
import pandas as pd
from comet import download_model, load_from_checkpoint

# 1. Locate Predictions File
pred_candidates = [
    "/content/drive/MyDrive/Inference/IndicTrans2_Results/indictrans2_annotated_predictions.csv",
    "/content/drive/MyDrive/IndicTrans2_Results/indictrans2_annotated_predictions.csv",
    "/content/drive/MyDrive/Inference/IndicTrans2_Results/predictions.csv",
    "/content/indictrans2_annotated_predictions.csv",
    os.path.join(os.getcwd(), "Inference", "IndicTrans2_Results", "indictrans2_annotated_predictions.csv"),
    os.path.join(os.getcwd(), "IndicTrans2_Results", "indictrans2_annotated_predictions.csv"),
]

pred_csv = next((p for p in pred_candidates if os.path.exists(p)), None)

if pred_csv and os.path.exists(pred_csv):
    print(f"Loading predictions from: {pred_csv}")
    df_eval = pd.read_csv(pred_csv, encoding="utf-8-sig")
    en_col = next((c for c in df_eval.columns if "english" in c.lower()), df_eval.columns[0])
    ref_col = next((c for c in df_eval.columns if "malayalam" in c.lower() and "pred" not in c.lower() and "indic" not in c.lower()), df_eval.columns[1])
    mt_col = next((c for c in df_eval.columns if "pred" in c.lower() or "indic" in c.lower()), df_eval.columns[-1])

    src_list = df_eval[en_col].astype(str).str.strip().tolist()
    ref_list = df_eval[ref_col].astype(str).str.strip().tolist()
    mt_list = df_eval[mt_col].astype(str).str.strip().tolist()
elif "en_sents" in globals() and "predictions" in globals() and "reference" in globals():
    src_list = en_sents
    mt_list = predictions
    ref_list = reference
    pred_csv = "/content/drive/MyDrive/Inference/IndicTrans2_Results/indictrans2_annotated_predictions.csv"
else:
    raise FileNotFoundError("Could not locate predictions CSV file to compute COMET.")

print(f"Evaluating {len(src_list)} sentence pairs with COMET...")

# 2. Load COMET model
print("Downloading and loading model: Unbabel/wmt22-comet-da...")
model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(model_path)

# 3. Run Inference
comet_data = [{"src": s, "mt": m, "ref": r} for s, m, r in zip(src_list, mt_list, ref_list)]
comet_output = comet_model.predict(comet_data, batch_size=8, gpus=1 if torch.cuda.is_available() else 0)
comet_score = round(float(comet_output.system_score), 4)

print("\n" + "=" * 60)
print(f"  INDICTRANS2 COMET SCORE (wmt22-comet-da): {comet_score:.4f}")
print("=" * 60)

# 4. Update Metrics CSV & TXT permanently across all result paths
save_dirs = [
    os.path.dirname(pred_csv) if pred_csv else None,
    "/content/drive/MyDrive/Inference/IndicTrans2_Results",
    "/content/drive/MyDrive/IndicTrans2_Results",
    "/content"
]

updated_df = None
for sdir in [d for d in save_dirs if d and os.path.exists(d)]:
    for base_m in ["indictrans2_annotated_metrics", "metrics"]:
        m_csv = os.path.join(sdir, f"{base_m}.csv")
        m_txt = os.path.join(sdir, f"{base_m}.txt")
        if os.path.exists(m_csv):
            df_m = pd.read_csv(m_csv)
            df_m["COMET"] = comet_score
            df_m.to_csv(m_csv, index=False)
            updated_df = df_m
            print(f"Updated {m_csv}")
        if os.path.exists(m_txt):
            with open(m_txt, "r", encoding="utf-8") as f:
                lines = f.readlines()
            newlines = [l for l in lines if not l.startswith("COMET:")]
            newlines.append(f"COMET: {comet_score}\n")
            with open(m_txt, "w", encoding="utf-8") as f:
                f.writelines(newlines)
            print(f"Updated {m_txt}")

if updated_df is not None:
    display(updated_df)
